In [ ]:
# Parameters — edit these before running
REPO_URL    = "https://github.com/Dev252001/ham10000-lesion-eval.git"
REPO_BRANCH = "main"  # set to another branch to test before merging
SEED        = 42
SPLIT_MODE  = "lesion"  # "image" is for the leakage comparison

In [ ]:
# Setup: mount Drive, clone / checkout / pull repo, install extras
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, subprocess, sys
from pathlib import Path

DRIVE_DIR  = Path("/content/drive/MyDrive/ham10000_project")
REPO_DIR   = DRIVE_DIR / "ham10000-lesion-eval"
CKPT_DIR   = DRIVE_DIR / "checkpoints"
DATA_DIR   = Path("/content/ham10000_data")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR / "src"))

# Install extras only — torch is pre-installed on Colab T4
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pandas", "scikit-learn", "matplotlib", "Pillow",
     "grad-cam", "kaggle"],
    check=True,
)

In [ ]:
# Kaggle credentials — upload kaggle.json via the file widget
# The file is never written into the repository.
from google.colab import files
import json
from pathlib import Path

kaggle_cfg = Path.home() / ".kaggle" / "kaggle.json"
if not kaggle_cfg.exists():
    print("Upload your kaggle.json now:")
    uploaded = files.upload()
    kaggle_cfg.parent.mkdir(parents=True, exist_ok=True)
    kaggle_cfg.write_bytes(list(uploaded.values())[0])
    kaggle_cfg.chmod(0o600)
else:
    print("kaggle.json already present.")

In [ ]:
# Download dataset (cached on Drive)
import shutil

DATA_CACHE = DRIVE_DIR / "ham10000_raw"
if not DATA_CACHE.exists():
    import kaggle
    kaggle.api.authenticate()
    kaggle.api.dataset_download_files(
        "kmader/skin-cancer-mnist-ham10000",
        path=str(DATA_CACHE),
        unzip=True,
    )
else:
    print("Dataset cache found on Drive.")

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
shutil.copytree(str(DATA_CACHE), str(DATA_DIR))
print(f"Data ready at {DATA_DIR}")

In [ ]:
# Load metadata and compute dataset statistics
import json
from skinlesion.data import load_metadata, dataset_stats

df = load_metadata(DATA_DIR)
print(f"Total images: {len(df)}")
print(f"Unique lesions: {df['lesion_id'].nunique()}")

stats = dataset_stats(df)
stats_path = REPO_DIR / "outputs" / "dataset_stats.json"
stats_path.parent.mkdir(parents=True, exist_ok=True)
stats_path.write_text(
    json.dumps(stats, indent=2), encoding="utf-8"
)
print(f"Saved {stats_path}")

In [ ]:
# Generate splits and audits for both modes at SEED
from skinlesion.splits import make_split, split_manifest, save_manifest, audit_split

splits_dir = REPO_DIR / "outputs" / "splits"
audits_dir = splits_dir  # audits saved alongside manifests

for mode in ("lesion", "image"):
    train, val, test = make_split(df, split_mode=mode, seed=SEED)
    manifest = split_manifest(train, val, test)
    save_manifest(manifest, splits_dir, split_mode=mode, seed=SEED)
    audit = audit_split(manifest)
    import json as _json
    audit_path = splits_dir / f"audit_{mode}_seed{SEED}.json"
    audit_path.write_text(_json.dumps(audit, indent=2), encoding="utf-8")
    print(f"[{mode}] train={len(train)}, val={len(val)}, test={len(test)}")

In [ ]:
# Class distribution figure
import matplotlib.pyplot as plt
import pandas as pd
from skinlesion.data import CLASSES

counts = df.groupby("dx").size().reindex(CLASSES, fill_value=0)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(CLASSES, counts.values, color="steelblue")
ax.set_xlabel("Class")
ax.set_ylabel("Image count")
ax.set_title("Image count per class")
fig.tight_layout()
fig_path = REPO_DIR / "outputs" / "figures" / "class_distribution.png"
fig.savefig(fig_path, dpi=100)
plt.close(fig)
print(f"Saved {fig_path}")

In [ ]:
# Images vs unique lesions per class figure
fig, ax = plt.subplots(figsize=(8, 4))
lesion_counts = df.groupby("dx")["lesion_id"].nunique().reindex(CLASSES, fill_value=0)
x = range(len(CLASSES))
ax.bar([i - 0.2 for i in x], counts.values, width=0.4, label="Images")
ax.bar([i + 0.2 for i in x], lesion_counts.values, width=0.4, label="Unique lesions")
ax.set_xticks(list(x))
ax.set_xticklabels(CLASSES)
ax.set_ylabel("Count")
ax.set_title("Images vs unique lesions per class")
ax.legend()
fig.tight_layout()
fig_path2 = REPO_DIR / "outputs" / "figures" / "images_vs_lesions.png"
fig.savefig(fig_path2, dpi=100)
plt.close(fig)
print(f"Saved {fig_path2}")

In [ ]:
# Package outputs and download
from skinlesion.export import make_outputs_zip

zip_path = DRIVE_DIR / "outputs_01.zip"
make_outputs_zip(REPO_DIR, zip_path)
from google.colab import files
files.download(str(zip_path))
print("Unzip at the root of your local clone, then commit.")